# Assignment 2



## Section 2: Multi-Head Self-Attention with KV-Cache (11 pts)

In this section, we are going to complete a MHSA implementation optimized with KV-cache using the einsum notation. For the rest of this section, we refer to the batch size as b, sequence length as q or k, number of heads as n, and head hidden dimension as h. The input to the MHSA layer is the tensor x, and the query, key and value weights are denoted as self.w_q, self.w_k & self.w_v respectively

**Specifically, complete the missing code in the `forward` function in `CausalSelfAttention` marked by a "_".**

On a T4, the code should complete in ~20-25 seconds.


**Point Breakdown**
- Problem 1: 3 pts
- Problem 2: 3 pts
- Problem 3: 3 pts
- Problem 4: 2 pts

### Problem

In [1]:
import math
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        # For pedagogical purposes only, we initialize them naively.
        self.w_q = nn.Parameter(torch.randn(config.n_embd, config.n_head, config.n_embd // config.n_head, requires_grad=True))
        self.w_k = nn.Parameter(torch.randn(config.n_embd, config.n_head, config.n_embd // config.n_head, requires_grad=True))
        self.w_v = nn.Parameter(torch.randn(config.n_embd, config.n_head, config.n_embd // config.n_head, requires_grad=True))
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # causal mask to ensure that attention is only applied to the left in the input sequence
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x, kvcache=None):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        ### Problem 1: calculate query, key, values for all heads in batch #
        q = torch.einsum('bqd, dnh -> bqnh', x, self.w_q)
        k = torch.einsum('bkd, dnh -> bknh', x, self.w_k)
        v = torch.einsum('bkd, dnh -> bknh', x, self.w_v)
        ### END ############################################################

        ### Problem 2: Implement KV Cache ##################################
        if kvcache:
            prev_k, prev_v = kvcache
            k = torch.cat([prev_k, k], dim=1)
            v = torch.cat([prev_v, v], dim=1)

        new_kvcache = (k, v)
        curr_T = k.size(1)
        ### END ############################################################

        ### Problem 3: Perform QKT Matmul ##################################
        att = torch.einsum('bqnh, bknh -> bnqk', q, k)
        ### END ############################################################

        att = att / math.sqrt(k.size(-1))
        if kvcache:
            att = att.masked_fill(torch.ones_like(self.bias[:,:,:T,:curr_T]) == 0, float('-inf'))
        else:
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = att.to(torch.float32)
        att = F.softmax(att, dim=-1)
        att = att.to(x.dtype)
        att = self.attn_dropout(att)

        ### Problem 4: Perform AV Matmul ####################################
        y = torch.einsum('bnqk, bknh -> bqnh', att, v)
        y = y.contiguous().view(B, T, C)
        ### END ############################################################


        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y, new_kvcache

### Helpers

In [24]:
class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x, kvcache=None):
        attn_out, cache_ele = self.attn(self.ln_1(x), kvcache)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return x, cache_ele

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self, non_embedding=True):
        """
        Return the number of parameters in the model.
        For non-embedding count (default), the position embeddings get subtracted.
        The token embeddings would too, except due to the parameter sharing these
        params are actually used as weights in the final layer, so we include them.
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, kvcache=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)
        # print(f"Forwarding on device: {device}, transformer device: {next(self.transformer.parameters()).device}")

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (1, t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)

        if not kvcache:
            kvcache = [None] * self.config.n_layer
        else:
            x = x[:, [-1], :]

        new_kvcache = []
        for block, kvcache_block in zip(self.transformer.h, kvcache):
            x, cache_ele = block(x, kvcache=kvcache_block)
            new_kvcache.append(cache_ele)

        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        return logits, loss, new_kvcache

    def crop_block_size(self, block_size):
        # model surgery to decrease the block size if necessary
        # e.g. we may load the GPT2 pretrained model checkpoint (block size 1024)
        # but want to use a smaller block size for some smaller, simpler model
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])
        for block in self.transformer.h:
            if hasattr(block.attn, 'bias'):
                block.attn.bias = block.attn.bias[:,:,:block_size,:block_size]

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        # start with all of the candidate parameters
        param_dict = {pn: p for pn, p in self.named_parameters()}
        # filter out those that do not require grad
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
        # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
        # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
        # Create AdamW optimizer and use the fused version if it is available
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        print(f"using fused AdamW: {use_fused}")

        return optimizer

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        """ estimate model flops utilization (MFU) in units of A100 bfloat16 peak FLOPS """
        # first estimate the number of flops we do per iteration.
        # see PaLM paper Appendix B as ref: https://arxiv.org/abs/2204.02311
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd//cfg.n_head, cfg.block_size
        flops_per_token = 6*N + 12*L*H*Q*T
        flops_per_fwdbwd = flops_per_token * T
        flops_per_iter = flops_per_fwdbwd * fwdbwd_per_iter
        # express our flops throughput as ratio of A100 bfloat16 peak flops
        flops_achieved = flops_per_iter * (1.0/dt) # per second
        flops_promised = 312e12 # A100 GPU bfloat16 peak flops is 312 TFLOPS
        mfu = flops_achieved / flops_promised
        return mfu

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        """
        kvcache = None
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _, kvcache = self(idx_cond, kvcache=kvcache)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

### Model Config

In [26]:
from contextlib import nullcontext
import torch
import time

# -----------------------------------------------------------------------------
start = "\n" # or "<|endoftext|>" or etc. Can also specify a file, use as: "FILE:prompt.txt"
max_new_tokens = 250 # number of tokens generated in each sample
temperature = 1.0 # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
top_k = 200 # retain only the top_k most likely tokens, clamp others to have 0 probability
seed = 1337
device = 'cuda' # examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1', etc.
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32' or 'bfloat16' or 'float16'
# -----------------------------------------------------------------------------

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cuda.matmul.allow_tf32 = True # allow tf32 on matmul
torch.backends.cudnn.allow_tf32 = True # allow tf32 on cudnn
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)
# print(f"using device: {device}, device type: {device_type}")

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = False # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster

# Instatiate new model.
cnfg = GPTConfig()
model = GPT(cnfg)
model.eval()
model.to(device)

number of parameters: 123.59M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50304, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_proj): Linear(in_features=768, out_features=768, bias=False)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=3072, out_features=768, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=768, out_features=50304, bias=False)
)

### Run Code

In [27]:
## First, we warmup the model. ##
print("Warming up the model...")
for i in range(5):
    warm = torch.randint(0, cnfg.vocab_size, (16, 200), device=device, dtype=torch.long)
    model(warm)

print("Starting generation and timing...")
# Next, run generation and timing.
with torch.inference_mode():
    ## We generate fake data first.
    s = 800
    x = torch.randint(low=0, high=cnfg.vocab_size-1, size=(16, 200), device=device, dtype=torch.long)
    cum_time = 0
    ## Start the timer.
    torch.cuda.synchronize()
    start_time = time.time()
    with ctx:
        y = model.generate(x, s, temperature=temperature, top_k=top_k)
    ## End the timer.
    torch.cuda.synchronize()
    end_time = time.time()
    cum_time += (end_time-start_time)
    print(f'prefill length: {200}, generation length: {s}, time taken: {cum_time:0.4f}')



Warming up the model...
Starting generation and timing...
prefill length: 200, generation length: 800, time taken: 6.9327


In [2]:
import torch, math
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Hyperparams you can match to your class ----
B, C = 1, 64           # batch, embed dim
n_head, head_dim = 8, 8
T_prefill, T_gen = 50, 12   # context length and tokens to "generate"
T_total = T_prefill + T_gen

# ---- Dummy module using your attention block ----
# Replace with your actual model that uses CausalSelfAttention internally.
class TinyBlock(torch.nn.Module):
    def __init__(self, attn):
        super().__init__()
        self.attn = attn
        self.proj = torch.nn.Linear(C, C, bias=False)
    def forward(self, x, kvcache=None):
        y, new_kvcache, curr_T = self.attn(x, kvcache=kvcache)
        y = self.proj(y)
        return y, new_kvcache, curr_T

# Construct your CausalSelfAttention(...) here
attn = CausalSelfAttention(
    n_head=n_head,
    head_dim=head_dim,
    n_embd=C,
    # ...whatever args your class needs...
).to(device)

blk = TinyBlock(attn).to(device).eval()

# ---- Fake token embeddings for a "sequence" ----
# In a real test you’d pass token IDs through an embedding first.
x_full = torch.randn(B, T_total, C, device=device)

# (A) Baseline: "no cache", recompute everything each time-step
#     For step t we run full prefix x_full[:, :t, :]
with torch.no_grad():
    baseline_step_logits = []
    for t in range(1, T_total+1):
        y, _, _ = blk(x_full[:, :t, :], kvcache=None)
        baseline_step_logits.append(y[:, -1, :].clone())  # last position

# (B) Cached: prefill, then incremental
with torch.no_grad():
    # 1) Prefill phase
    prefill_x = x_full[:, :T_prefill, :]
    _, kvcache, _ = blk(prefill_x, kvcache=None)

    # 2) Incremental phase: feed one token at a time of the continuation
    cached_step_logits = []
    # First, collect the baseline’s prefill last token output
    cached_step_logits.append(baseline_step_logits[T_prefill-1].clone())

    for t in range(T_prefill, T_total):
        x_t = x_full[:, t:t+1, :]  # shape (B, 1, C)
        y_t, kvcache, _ = blk(x_t, kvcache=kvcache)  # reuses cache
        cached_step_logits.append(y_t[:, -1, :].clone())

# Compare (A) vs (B): last-token outputs step by step
tol = 1e-5
max_errs = []
for t in range(T_total):
    a = baseline_step_logits[t]
    b = cached_step_logits[t] if t >= (T_prefill-1) else None
    if b is None:
        continue  # we start comparing at prefill's last step
    err = (a - b).abs().max().item()
    max_errs.append(err)

print("max per-step diff after prefill:", max(max_errs))
assert max(max_errs) < 1e-4, "Cached decoding diverges from baseline!"
print("✅ KV cache equivalence holds.")


TypeError: CausalSelfAttention.__init__() got an unexpected keyword argument 'n_head'